# 07 · Modelos não lineares

Random Forest, LSTM e BiLSTM com as features escolhidas na etapa 06. Os modelos preveem o **desvio em relação ao último dia** (alvo − `fim_vs_media`), e a previsão final é último dia + desvio. Assim, um modelo que não aprende nada fica igual ao último dia.

Hiperparâmetros escolhidos pelo Optuna só na validação (2010 a 2014), com retreino anual. No teste, as redes usam a média de 5 sementes.

In [ ]:
import sys, json
from pathlib import Path

if Path("/kaggle/input").exists():
    RAIZ = next(Path("/kaggle/input").rglob("src/soja/__init__.py")).parents[2]
else:
    RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from soja import config, avaliacao, experimento, otimizacao

ETAPA = "07_modelos"
PASTA = config.pasta_resultados(ETAPA)
plt.rcParams.update({"figure.figsize": (11, 3.5), "axes.grid": True, "grid.alpha": .3})

tabela = pd.read_csv(config.ler_resultado("05_tabela", "tabela_mensal.csv"), index_col="mes")
tabela.index = pd.PeriodIndex(tabela.index, freq="M")
COLUNAS = json.loads(config.ler_resultado("06_features", "config.json").read_text())["colunas_finais"]
tabela["desvio"] = tabela["alvo"] - tabela["fim_vs_media"]
print(len(COLUNAS), "features:", COLUNAS)

PERIODOS = {"validacao": (config.VALIDACAO_INICIO, config.VALIDACAO_FIM), "teste": (config.TESTE_INICIO, config.TESTE_FIM)}

def avaliar(nome, desvio_previsto, periodo):
    meses = desvio_previsto.index
    prev = tabela.loc[meses, "fim_vs_media"] + desvio_previsto
    refs = {"passeio": pd.Series(0.0, index=meses), "ultimo_dia": tabela.loc[meses, "fim_vs_media"]}
    return prev.rename("previsao"), {"modelo": nome, "periodo": periodo, "meses": len(meses),
                                     **avaliacao.resumo(prev, tabela.loc[meses, "alvo"], refs)}

## 1. Ridge sobre o desvio (referência linear)

In [ ]:
linhas, previsoes = [], {}
for periodo, (ini, fim) in PERIODOS.items():
    desvio = experimento.janela_expansiva(tabela.assign(alvo=tabela["desvio"]), COLUNAS, ini, fim)
    previsoes[("ridge", periodo)], r = avaliar("ridge", desvio, periodo)
    linhas.append(r)

real = tabela.loc[previsoes[("ridge", "teste")].index, "alvo"]
plt.plot(real.index.to_timestamp(), real, "k", lw=.8, label="alvo")
plt.plot(real.index.to_timestamp(), previsoes[("ridge", "teste")], "tab:red", label="Ridge")
plt.legend(); plt.title("Ridge no teste"); plt.show()
pd.DataFrame(linhas).round(3)

## 2. Optuna na validação

In [ ]:
estudos, melhores = {}, {}
for tipo in ["rf", "lstm", "bilstm"]:
    estudos[tipo] = otimizacao.buscar(tipo, tabela, COLUNAS, "desvio", config.N_TRIALS)
    melhores[tipo] = estudos[tipo].best_params
    print(f"{tipo}: RMSE {estudos[tipo].best_value:.5f} com {melhores[tipo]}")

fig, ax = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for a, (tipo, estudo) in zip(ax, estudos.items()):
    valores = [t.value for t in estudo.trials]
    a.plot(valores, "o", color="lightgray"); a.plot(np.minimum.accumulate(valores), color="tab:red")
    a.set_title(f"{tipo}: RMSE na validação por tentativa")
plt.tight_layout(); plt.show()

## 3. Teste

In [ ]:
for tipo, params in melhores.items():
    for periodo, (ini, fim) in PERIODOS.items():
        desvio = otimizacao.prever(tipo, params, tabela, COLUNAS, "desvio", ini, fim, config.SEMENTES)
        previsoes[(tipo, periodo)], r = avaliar(tipo, desvio, periodo)
        linhas.append(r)
resumo = pd.DataFrame(linhas)

teste = resumo[resumo["periodo"] == "teste"].set_index("modelo")
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
teste[["rmse_rel_passeio", "rmse_rel_ultimo_dia"]].plot.bar(ax=ax[0], rot=0, title="RMSE no teste ÷ referência")
ax[0].axhline(1, c="k", lw=.8); ax[0].set_ylim(.6, 1.2)
ultimo = tabela.loc[real.index, "fim_vs_media"]
for tipo in teste.index:
    p = previsoes[(tipo, "teste")].reindex(real.index)
    ax[1].plot(real.index.to_timestamp(), ((real - ultimo) ** 2 - (real - p) ** 2).cumsum(), label=tipo)
ax[1].axhline(0, c="k", lw=.7); ax[1].legend(); ax[1].set_title("Vantagem acumulada contra o último dia")
plt.tight_layout(); plt.savefig(PASTA / "teste.png", dpi=120); plt.show()
resumo.round(3)

## 4. Salvando

In [ ]:
resumo.to_csv(PASTA / "resumo.csv", index=False)

prev_df = pd.concat({f"{m}|{p}": s for (m, p), s in previsoes.items()}, names=["chave", "mes"]).reset_index()
prev_df[["modelo", "periodo"]] = prev_df["chave"].str.split("|", expand=True)
prev_df["alvo"] = tabela.loc[pd.PeriodIndex(prev_df["mes"]), "alvo"].values
prev_df["mes"] = prev_df["mes"].astype(str)
prev_df[["modelo", "periodo", "mes", "previsao", "alvo"]].to_csv(PASTA / "previsoes.csv", index=False)

resumo[["modelo", "periodo", "rmse_rel_passeio", "p_passeio", "rmse_rel_ultimo_dia", "p_ultimo_dia"]].to_csv(PASTA / "diebold_mariano.csv", index=False)
(PASTA / "hiperparametros.json").write_text(json.dumps(melhores, indent=2))
pd.concat({t: e.trials_dataframe() for t, e in estudos.items()}, names=["modelo"]).to_csv(PASTA / "trials.csv")
config.salvar_config(PASTA, colunas=COLUNAS, alvo_dos_modelos="alvo - fim_vs_media", retreino="anual (Ridge: mensal)",
                     sementes=config.SEMENTES, n_trials=config.N_TRIALS,
                     validacao=[config.VALIDACAO_INICIO, config.VALIDACAO_FIM])
print(sorted(p.name for p in PASTA.iterdir()))